# CP5 · Notebook 02 — Setup

Verifica versiones + smoke test del entorno + smoke test de stable-baselines3. ~1 min.

In [1]:
import sys, platform
print(f'Python: {sys.version.split()[0]}  ({platform.system()} {platform.machine()})')
assert sys.version_info >= (3, 10)

Python: 3.14.6  (Darwin arm64)


In [2]:
import numpy as np, torch, gymnasium as gym, highway_env
import stable_baselines3 as sb3
print(f'numpy {np.__version__}')
print(f'torch {torch.__version__}  (cuda: {torch.cuda.is_available()})')
print(f'gymnasium {gym.__version__}')
print(f'highway-env {highway_env.__version__}')
print(f'stable-baselines3 {sb3.__version__}')

objc[17121]: Class SDL_RumbleMotor is implemented in both /opt/homebrew/Cellar/sdl2/2.32.2/lib/libSDL2-2.0.0.dylib (0x116bb8bf8) and /private/tmp/claude-501/-Users-erlantzmarcos-Projects-AIC/99882926-c5dd-4419-9d4b-4d9562471efd/scratchpad/venv-cp5/lib/python3.14/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x1170b89c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[17121]: Class SDL_RumbleContext is implemented in both /opt/homebrew/Cellar/sdl2/2.32.2/lib/libSDL2-2.0.0.dylib (0x116bb8c48) and /private/tmp/claude-501/-Users-erlantzmarcos-Projects-AIC/99882926-c5dd-4419-9d4b-4d9562471efd/scratchpad/venv-cp5/lib/python3.14/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x1170b8a18). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[17121]: Class SDLApplication is implemented in both /opt/homebrew/Cellar/sdl2/2.32.2/lib/libSDL2-2.0.0.dylib 

numpy 2.5.1
torch 2.13.0  (cuda: False)
gymnasium 1.3.0
highway-env 1.12.0
stable-baselines3 2.9.0


## Crear highway-v0 con DiscreteMetaAction

In [3]:
env = gym.make('highway-v0', config={
    'observation': {
        'type': 'Kinematics',
        'vehicles_count': 5,
        'features': ['presence', 'x', 'y', 'vx', 'vy'],
        'normalize': True,
    },
    'action': {'type': 'DiscreteMetaAction'},
    'lanes_count': 4,
    'vehicles_count': 20,
    'duration': 40,
    'policy_frequency': 5,
})
obs, _ = env.reset(seed=0)
print(f'Observation space: {env.observation_space}')
print(f'Action space:      {env.action_space}')
print(f'Acciones disponibles: {{i: n for n, i in env.unwrapped.action_type.actions_indexes.items()}}')  # dict id -> name
print(f'\nPrimera obs shape: {obs.shape}')
print(f'Primera obs (5 vehículos × 5 features):\n{obs}')
env.close()

Observation space: Box(-inf, inf, (5, 5), float32)
Action space:      Discrete(5)
Acciones disponibles: {i: n for n, i in env.unwrapped.action_type.actions_indexes.items()}

Primera obs shape: (5, 5)
Primera obs (5 vehículos × 5 features):
[[ 1.          0.8873327   0.75        0.3125      0.        ]
 [ 1.          0.09073734 -0.25       -0.04846349  0.        ]
 [ 1.          0.20118086 -0.25       -0.02725116  0.        ]
 [ 1.          0.31662506  0.         -0.01493478  0.        ]
 [ 1.          0.42161888 -0.5        -0.04874054  0.        ]]


## Smoke test stable-baselines3

In [4]:
from stable_baselines3 import DQN

env = gym.make('highway-v0', config={
    'observation': {'type': 'Kinematics', 'vehicles_count': 5, 'features': ['presence','x','y','vx','vy'], 'normalize': True},
    'action': {'type': 'DiscreteMetaAction'},
    'lanes_count': 4, 'vehicles_count': 20, 'duration': 40,
})

# DQN espera obs Flatten — usamos un wrapper trivial
from gymnasium.wrappers import FlattenObservation
env = FlattenObservation(env)

model = DQN('MlpPolicy', env, learning_rate=5e-4, buffer_size=10_000, learning_starts=200,
             batch_size=64, gamma=0.95, verbose=0)
print('DQN creado OK')
print(f'Política: {model.policy}')
model.learn(total_timesteps=200)
print('✅ Smoke test: DQN entrena 200 timesteps sin errores')
env.close()

DQN creado OK
Política: DQNPolicy(
  (q_net): QNetwork(
    (features_extractor): FlattenExtractor(
      (flatten): Flatten(start_dim=1, end_dim=-1)
    )
    (q_net): Sequential(
      (0): Linear(in_features=25, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=5, bias=True)
    )
  )
  (q_net_target): QNetwork(
    (features_extractor): FlattenExtractor(
      (flatten): Flatten(start_dim=1, end_dim=-1)
    )
    (q_net): Sequential(
      (0): Linear(in_features=25, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=5, bias=True)
    )
  )
)


✅ Smoke test: DQN entrena 200 timesteps sin errores


In [5]:
print('✅ Setup OK — listo para 03_entorno.ipynb')

✅ Setup OK — listo para 03_entorno.ipynb
